In [15]:
import pandas as pd
import numpy as np
import sqlite3
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import json
import copy
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行

# ==========================================
# 0. 自动设备检测
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# ==========================================
# 1. 外置数据预处理模块 (ETL & Feature Engineering)
# ==========================================
print(">>> [1/3] Loading and Preprocessing Data (Externalized)...")

micro_df = pd.read_parquet('./data/micro_data.parquet')
macro_df = pd.read_parquet('./data/macro_data.parquet')
micro_cols=micro_df.columns.drop(['mthcaldt', 'permno', 'mthret_lead1'])
macro_cols=macro_df.columns.drop('date')

# 1.1 日期对齐与合并
micro_df['mthcaldt'] = pd.to_datetime(micro_df['mthcaldt']) + pd.offsets.MonthEnd(0)
macro_df['date'] = pd.to_datetime(macro_df['date']) + pd.offsets.MonthEnd(0)

df = pd.merge(micro_df, macro_df, left_on='mthcaldt', right_on='date', how='inner')
df = df.sort_values(['permno', 'mthcaldt']).reset_index(drop=True)

# 1.2 目标收益率设定 (绝对禁止缩尾！)
df['target_ret_final'] = df['mthret_lead1']

# 1.3 缺失值预填充 (可在此处放置你的业务填充逻辑)
# 例如：使用截面均值填充微观特征缺失值
for feature in micro_cols:
    if feature in df.columns:
        df[feature] = df[feature].fillna(df.groupby('mthcaldt')[feature].transform('mean'))

# 1.4 微观特征：截面秩标准化 (Cross-Sectional Rank Normalization) [-1, 1]
print(" -> Applying Rank Normalization to Micro Features...")
micro_tensor_cols = []
for col in micro_cols:
    norm_col = f'{col}_norm'
    # 使用 pct=True 转化为 0~1，再映射到 -1~1
    df[norm_col] = df.groupby('mthcaldt')[col].transform(
        lambda x: (x.rank(pct=True) * 2) - 1
    ).fillna(0)  # 如果全截面缺失，填0(中性)
    micro_tensor_cols.append(norm_col)

# 1.5 宏观特征：时间序列 Z-score 标准化
print(" -> Applying Z-score Normalization to Macro Features...")
macro_tensor_cols = []
for col in macro_cols:
    norm_col = f'{col}_norm'
    df[norm_col] = (df[col] - df[col].mean()) / (df[col].std() + 1e-8)
    df[norm_col] = df[norm_col].fillna(0)
    macro_tensor_cols.append(norm_col)

# 1.6 清理最终的空值行并固化数据集
processed_df = df.dropna(subset=['target_ret_final'] + micro_tensor_cols + macro_tensor_cols).copy()
print(f"Data preprocessing complete! Final shape: {processed_df.shape}")
print(f"Micro Features: {len(micro_tensor_cols)}, Macro Features: {len(macro_tensor_cols)}")

# (可选) 你可以在这里将 processed_df 存为 Parquet:
# processed_df.to_parquet('./data/fully_processed_model_input.parquet')

Using device: mps
>>> [1/3] Loading and Preprocessing Data (Externalized)...
 -> Applying Rank Normalization to Micro Features...
 -> Applying Z-score Normalization to Macro Features...
Data preprocessing complete! Final shape: (1616304, 95)
Micro Features: 23, Macro Features: 22


In [17]:

# ==========================================
# 2. 轻量级时间窗口生成器
# ==========================================
class DataWindowGenerator:
    """
    仅负责按时间切分数据，向模型输送 Train/Val/Test 窗口
    """
    def __init__(self, df, date_col='mthcaldt'):
        self.df = df
        self.date_col = date_col
        
    def expanding_window_generator(self, initial_train_years=20, val_years=2, test_years=1):
        dates = np.sort(self.df[self.date_col].unique())
        initial_train_months = initial_train_years * 12
        val_months = val_years * 12
        test_months = test_years * 12
        
        start_idx = 0
        current_split_idx = initial_train_months
        
        while current_split_idx + val_months + test_months <= len(dates):
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data = self.df[self.df[self.date_col].isin(val_dates)]
            test_data = self.df[self.df[self.date_col].isin(test_dates)]
            
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            current_split_idx += test_months

# ==========================================
# 3. 结构化 MDN (Structured MoE-MDN)
# ==========================================
class StructuredMDN(nn.Module):
    def __init__(self, macro_dim, micro_dim, hidden_dim=64, num_components=3):
        super(StructuredMDN, self).__init__()
        self.num_components = num_components
        
        # 通道 A: Macro Network (门控网络) -> 预测机制权重 pi
        self.macro_net = nn.Sequential(
            nn.Linear(macro_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ELU(),
            nn.Linear(hidden_dim // 2, num_components)
        )
        
        # 通道 B: Micro Network (专家网络) -> 预测特质 mu 和 sigma
        self.micro_extractor = nn.Sequential(
            nn.Linear(micro_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ELU()
        )
        
        self.mu_head = nn.Linear(hidden_dim // 2, num_components)
        self.sigma_head = nn.Linear(hidden_dim // 2, num_components)

    def forward(self, x_macro, x_micro):
        pi_logits = self.macro_net(x_macro)
        h_micro = self.micro_extractor(x_micro)
        mu = self.mu_head(h_micro)
        sigma = nn.functional.softplus(self.sigma_head(h_micro)) + 1e-6 
        return pi_logits, mu, sigma
    
# ==========================================
# 4. 稳健的 NLL 损失函数 (支持时间衰减权重)
# ==========================================
def mdn_nll_loss_weighted(pi_logits, mu, sigma, target, weights):
    """
    weights: shape (batch_size, 1)，距离现在越近权重大
    """
    log_pi = torch.log_softmax(pi_logits, dim=-1)
    normal_dist = torch.distributions.Normal(mu, sigma)
    log_normal = normal_dist.log_prob(target.expand_as(mu))
    log_mix = log_pi + log_normal
    
    # 每个样本基础的 NLL loss
    loss = -torch.logsumexp(log_mix, dim=-1) 
    
    # 乘以时间衰减权重后求平均
    # 确保 weights 的 shape 与 loss 对齐
    weighted_loss = loss * weights.squeeze()
    
    return weighted_loss.mean()

# ==========================================
# 5. 双通道训练循环 (支持 Warm Start 与 Weights)
# ==========================================
def train_structured_mdn(model,  # 【修改点1】模型从外部传入，实现热启动
                         X_train_macro, X_train_micro, y_train, w_train, # 【修改点2】加入训练权重
                         X_val_macro, X_val_micro, y_val, w_val,         # 【修改点2】加入验证权重
                         epochs=500, batch_size=512, lr=1e-3, patience=20):
    
    # 每次进入新窗口，重新初始化 Optimizer，清除历史动量缓存，防止陷入局部最优
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # 将权重 w_train 和 w_val 打包进 TensorDataset
    tr_dataset = TensorDataset(torch.FloatTensor(X_train_macro), 
                               torch.FloatTensor(X_train_micro), 
                               torch.FloatTensor(y_train).unsqueeze(1),
                               torch.FloatTensor(w_train).unsqueeze(1))
    
    va_dataset = TensorDataset(torch.FloatTensor(X_val_macro), 
                               torch.FloatTensor(X_val_micro), 
                               torch.FloatTensor(y_val).unsqueeze(1),
                               torch.FloatTensor(w_val).unsqueeze(1))
    
    train_loader = DataLoader(tr_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(va_dataset, batch_size=batch_size, shuffle=False)
    
    best_val_loss = float('inf')
    best_model_state = None
    epochs_no_improve = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for b_macro, b_micro, b_y, b_w in train_loader:
            b_macro, b_micro, b_y, b_w = b_macro.to(device), b_micro.to(device), b_y.to(device), b_w.to(device)
            optimizer.zero_grad()
            
            pi_logits, mu, sigma = model(b_macro, b_micro)
            # 【修改点3】调用带权重的 Loss
            loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, b_y, b_w)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * b_macro.size(0)
            
        train_loss /= len(train_loader.dataset)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for b_macro, b_micro, b_y, b_w in val_loader:
                b_macro, b_micro, b_y, b_w = b_macro.to(device), b_micro.to(device), b_y.to(device), b_w.to(device)
                pi_logits, mu, sigma = model(b_macro, b_micro)
                loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, b_y, b_w)
                val_loss += loss.item() * b_macro.size(0)
        val_loss /= len(val_loader.dataset)
            
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break
                
    model.load_state_dict(best_model_state)
    return model, best_val_loss


# ==========================================
# 6. 启动模型训练流水线
# ==========================================
def calculate_time_decay_weights(df_dates, half_life_months=36):
    """
    计算基于半衰期的指数衰减权重。
    参数：
    df_dates: pd.Series, 包含 mthcaldt 日期
    half_life_months: 半衰期(月)。默认 36 个月(3年)权重减半。
    """
    max_date = df_dates.max()
    # 计算每个样本距离该窗口最新日期的月数差
    months_diff = (max_date.year - df_dates.dt.year) * 12 + (max_date.month - df_dates.dt.month)
    
    # 衰减因子 lambda: (0.5)^(1/half_life)
    decay_factor = np.power(0.5, 1.0 / half_life_months)
    
    # 计算基础权重
    weights = np.power(decay_factor, months_diff).values
    
    # 【极度关键】归一化权重，使得权重的均值为 1。
    # 如果不归一化，整个 Loss 的绝对值会变小，导致梯度的步长变相缩小！
    weights = weights / weights.mean() 
    return weights

if __name__ == "__main__":
    
    print("\n>>> [2/3] Initializing Data Window Generator...")
    
    generator = DataWindowGenerator(df=processed_df, date_col='mthcaldt')
    window_gen = generator.expanding_window_generator(initial_train_years=15, val_years=2, test_years=1)
    
    all_oos_predictions = []
    
    # 【修改点4：热启动核心】在循环外全局初始化模型，让它继承历史记忆！
    best_k = 3
    macro_dim = len(macro_tensor_cols)
    micro_dim = len(micro_tensor_cols)
    global_model = StructuredMDN(macro_dim, micro_dim, num_components=best_k).to(device)
    
    print("\n>>> [3/3] Starting Expanding Window Structured MDN Training (Warm Start & Time Decay)...")
    for window_idx, (info, train_df, val_df, test_df) in enumerate(window_gen):
        print(f"\n[Window {window_idx + 1}] Train: {info['train'][0]}~{info['train'][1]} | Test: {info['test'][0]}~{info['test'][1]}")
        
        # --- 计算时间衰减权重 ---
        w_train = calculate_time_decay_weights(train_df['mthcaldt'], half_life_months=36)
        w_val = calculate_time_decay_weights(val_df['mthcaldt'], half_life_months=12) # 验证集更看重近期，半衰期设短一点
        
        X_tr_mac, X_tr_mic = train_df[macro_tensor_cols].values, train_df[micro_tensor_cols].values
        Y_tr = train_df['target_ret_final'].values
        
        X_va_mac, X_va_mic = val_df[macro_tensor_cols].values, val_df[micro_tensor_cols].values
        Y_va = val_df['target_ret_final'].values
        
        X_te_mac, X_te_mic = test_df[macro_tensor_cols].values, test_df[micro_tensor_cols].values
        
        # --- 动态学习率 ---
        # 第一轮窗口是从零开始(Cold Start)，学习率正常；
        # 后续窗口由于有 Warm Start，相当于微调，学习率降低防止破坏已有知识
        current_lr = 1e-3 if window_idx == 0 else 1e-4
        
        global_model, val_loss = train_structured_mdn(
            model=global_model,     # 传入全局模型
            X_train_macro=X_tr_mac, X_train_micro=X_tr_mic, y_train=Y_tr, w_train=w_train,
            X_val_macro=X_va_mac, X_val_micro=X_va_mic, y_val=Y_va, w_val=w_val,

            lr=current_lr           # 使用动态学习率
        )
        
        # --- 样本外预测 ---
        global_model.eval()
        with torch.no_grad():
            pi_logits, mu, sigma = global_model(
                torch.FloatTensor(X_te_mac).to(device), 
                torch.FloatTensor(X_te_mic).to(device)
            )
            pi = torch.softmax(pi_logits, dim=-1)
            
        preds_df = test_df[['permno', 'mthcaldt', 'target_ret_final']].copy()
        preds_df['pi_vec'] = [json.dumps(vec.tolist()) for vec in pi.cpu().numpy()]
        preds_df['mu_vec'] = [json.dumps(vec.tolist()) for vec in mu.cpu().numpy()]
        preds_df['sigma_vec'] = [json.dumps(vec.tolist()) for vec in sigma.cpu().numpy()]
        all_oos_predictions.append(preds_df)
        
        # 测试用，仅跑 1 个窗口
        # if window_idx == 0: break

    print("\n>>> Pipeline complete. You can now use Monte Carlo on (pi, mu, sigma) to compute ES_5%!")


>>> [2/3] Initializing Data Window Generator...

>>> [3/3] Starting Expanding Window Structured MDN Training (Warm Start & Time Decay)...

[Window 1] Train: 1990-01~2004-12 | Test: 2007-01~2007-12


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from scipy.stats.mstats import winsorize
import warnings
warnings.filterwarnings('ignore')

class AssetPricingPipeline:
    def __init__(self, db_path='tail_risk_data.db'):
        self.db_path = db_path
        self.df = None
        self.feature_cols = []
        
    def load_and_build_features(self):
        """
        从 SQLite 极速读取，并执行严格的特征工程和目标标准化
        """
        print("Loading data from SQLite database...")
        with sqlite3.connect(self.db_path) as conn:
            # SQLite 存储的时间是字符串，使用 parse_dates 直接转换为 datetime
            df_micro = pd.read_sql("SELECT * FROM micro_data", conn, parse_dates=['date', 'datadate', 'linkdt', 'linkenddt'])
            df_macro = pd.read_sql("SELECT * FROM macro_data", conn, parse_dates=['date'])
        
        print("Executing Feature Engineering...")
        # 确保排序，这对时序计算至关重要
        df = df_micro.sort_values(['permno', 'date']).reset_index(drop=True)
        
        # --- 1. 底层截面特征 ---
        df['Size'] = np.log(df['me'])
        df['BM'] = df['ceq'] / df['me'] 
        df['BM'] = np.where(df['BM'] < 0, np.nan, df['BM'])
        df['OP'] = (df['revt'] - df['cogs'].fillna(0) - df['xsga'].fillna(0) - df['xint'].fillna(0)) / df['ceq']
        
        df['at_lag1'] = df.groupby('permno')['at'].shift(1) 
        df['INV'] = (df['at'] - df['at_lag1']) / df['at_lag1']
        df['MOM1m'] = df.groupby('permno')['ret'].shift(1)
        
        df['log_ret'] = np.log(1 + df['ret'])
        df['MOM12m'] = df.groupby('permno')['log_ret'].apply(
            lambda x: x.shift(2).rolling(window=11).sum()
        ).reset_index(level=0, drop=True)
        df['MOM12m'] = np.exp(df['MOM12m']) - 1
        
        # 与宏观数据合并
        df = pd.merge(df, df_macro, on='date', how='inner')
        
        # --- 2. 目标收益率构建 (Target Normalization) ---
        # 预测下一期收益率
        df['target_ret_raw'] = df.groupby('permno')['ret'].shift(-1)
        
        # 用 VIX 缩放目标收益率 (消除宏观异方差)
        df['target_ret_scaled'] = df['target_ret_raw'] / (df['VIX'] / 100 / np.sqrt(12))
        
        # 截面缩尾 (Winsorize) - 防止极值破坏 MDN 的 NLL 损失, Winsorize 是将
        def cs_winsorize(group):
            if len(group.dropna()) > 50:
                return winsorize(group, limits=[0.01, 0.01])
            return group
        
        df['target_ret_final'] = df.groupby('date')['target_ret_scaled'].transform(cs_winsorize)
        
        # --- 3. 张量特征展开与截面秩标准化 ---
        micro_feats = ['Size', 'BM', 'OP', 'INV', 'MOM1m', 'MOM12m']
        macro_feats = ['TERM', 'DEF'] 
        
        tensor_cols = []
        for micro in micro_feats:
            norm_col = f'{micro}_norm'
            df[norm_col] = df.groupby('date')[micro].transform(
                lambda x: (x.rank() - 1) / (len(x.dropna()) - 1) * 2 - 1
            ).fillna(0) 
            tensor_cols.append(norm_col)
            
            # 张量交互: 允许 Beta 时变
            for macro in macro_feats:
                interact_name = f'{norm_col}_x_{macro}'
                df[interact_name] = df[norm_col] * df[macro]
                tensor_cols.append(interact_name)
                
        self.feature_cols = tensor_cols
        # 清除无法作为训练集的行（缺乏目标或特征）
        self.df = df.dropna(subset=['target_ret_final'] + self.feature_cols)
        print(f"Pipeline ready. Total Tensor Features: {len(self.feature_cols)}")

    def expanding_window_generator(self, initial_train_years=20, val_years=2, test_years=1):
        """
        纯样本外扩展窗口生成器
        """
        dates = np.sort(self.df['date'].unique())
        
        initial_train_months = initial_train_years * 12
        val_months = val_years * 12
        test_months = test_years * 12
        
        start_idx = 0
        current_split_idx = initial_train_months
        
        while current_split_idx + val_months + test_months <= len(dates):
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            train_data = self.df[self.df['date'].isin(train_dates)]
            val_data = self.df[self.df['date'].isin(val_dates)]
            test_data = self.df[self.df['date'].isin(test_dates)]
            
            # 修复：使用 pd.Timestamp() 包装 numpy.datetime64 对象
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            
            yield window_info, train_data, val_data, test_data
            current_split_idx += test_months


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import sqlite3
import json
import copy
from tqdm import tqdm

# ==========================================
# 0. 自动设备检测 (支持 M1 GPU, Nvidia GPU 和 CPU)
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# ==========================================
# 1. 混合密度网络 (MDN) 核心架构
# ==========================================
class MDN(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_components=3):
        super(MDN, self).__init__()
        self.num_components = num_components
        
        # 共享特征提取层 (Shared Representation)
        # 考虑到金融数据的低信噪比，加入 Dropout 和 BatchNorm 防止严重过拟合
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ELU(), # ELU 比 ReLU 在处理金融特征时更能保留负向信号，且避免神经元死亡
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(0.2)
        )
        
        # 三个独立的多头输出层 (分别输出 权重、均值、方差)
        self.pi_head = nn.Linear(hidden_dim // 2, num_components)
        self.mu_head = nn.Linear(hidden_dim // 2, num_components)
        self.sigma_head = nn.Linear(hidden_dim // 2, num_components)

    def forward(self, x):
        h = self.feature_extractor(x)
        
        # 1. 权重 pi: 不在这里做 Softmax，我们在 loss 函数里用 log_softmax 保证数值稳定
        pi_logits = self.pi_head(h) 
        
        # 2. 均值 mu: 直接输出，无约束
        mu = self.mu_head(h)
        
        # 3. 标准差 sigma: 使用 ELU + 1 + epsilon 保证严格为正，且比 exp() 更不容易爆炸
        # EPSILON (1e-6) 是防爆破的关键底线
        sigma = nn.functional.elu(self.sigma_head(h)) + 1.0 + 1e-6 
        
        return pi_logits, mu, sigma

# ==========================================
# 2. 负对数似然损失函数 (NLL Loss)
# ==========================================
def mdn_nll_loss(pi_logits, mu, sigma, target):
    """
    使用 Log-Sum-Exp 技巧计算 NLL，绝对杜绝 NaN
    target: shape (batch_size, 1)
    """
    # 将 pi_logits 转化为 log(pi)
    log_pi = torch.log_softmax(pi_logits, dim=-1) # (batch, K)
    
    # 构造标准高斯分布对象 (利用 PyTorch 自带的 Normal 分布算 log_prob 更稳定)
    normal_dist = torch.distributions.Normal(mu, sigma)
    
    # 计算 log(N(y | mu, sigma))
    # target 扩展为 (batch, K) 以便与 mu, sigma 对齐计算
    log_normal = normal_dist.log_prob(target.expand_as(mu))
    
    # 混合概率的对数: log(pi * N) = log(pi) + log(N)
    log_mix = log_pi + log_normal
    
    # 核心技巧: log(sum(exp(log_mix))) 计算总体概率的对数
    loss = -torch.logsumexp(log_mix, dim=-1) 
    
    return loss.mean()
# ==========================================
# 3. 模型训练与 Early Stopping 闭环
# ==========================================
def train_mdn(X_train, y_train, X_val, y_val, k_components, epochs=999, batch_size=256, lr=1e-3, patience=20):
    """
    训练单个特定 K 值的 MDN 模型
    """
    input_dim = X_train.shape[1]
    # 模型移动到设备
    model = MDN(input_dim=input_dim, num_components=k_components).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4) # AdamW 加入权重衰减防过拟合
    
    # 转换为 PyTorch Tensors
    X_tr, y_tr = torch.FloatTensor(X_train), torch.FloatTensor(y_train).unsqueeze(1)
    X_va, y_va = torch.FloatTensor(X_val).to(device), torch.FloatTensor(y_val).unsqueeze(1).to(device)
    
    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    
    best_val_loss = float('inf')
    best_model_state = None
    epochs_no_improve = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch_x, batch_y in train_loader:
            # 数据移动到设备
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            pi_logits, mu, sigma = model(batch_x)
            loss = mdn_nll_loss(pi_logits, mu, sigma, batch_y)
            loss.backward()
            
            # 梯度裁剪 (Gradient Clipping)，防爆破的第二道防线
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            train_loss += loss.item() * batch_x.size(0)
            
        train_loss /= len(train_loader.dataset)
        
        # Validation
        model.eval()
        with torch.no_grad():
            pi_logits_val, mu_val, sigma_val = model(X_va)
            val_loss = mdn_nll_loss(pi_logits_val, mu_val, sigma_val, y_va).item()
            
        # Early Stopping 逻辑
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                # print(f"    [Early Stop] Epoch {epoch}: Val NLL = {best_val_loss:.4f}")
                break
                
    # 恢复最优权重
    model.load_state_dict(best_model_state)
    return model, best_val_loss

# ==========================================
# 4. 动态 K 值寻优 (Dynamic K Search)
# ==========================================
def find_best_mdn(X_train, y_train, X_val, y_val, k_candidates=[1, 3, 5, 7]):
    """
    在当前 Rolling Window 的验证集上，寻找表现最好的 K 个正态分布组合
    """
    best_k = None
    best_loss = float('inf')
    best_model = None
    
    print(f"    -> Searching best K in {k_candidates}...")
    for k in k_candidates:
        model, val_loss = train_mdn(X_train, y_train, X_val, y_val, k_components=k)
        print(f"       K={k} | Val NLL: {val_loss:.4f}")
        
        if val_loss < best_loss:
            best_loss = val_loss
            best_k = k
            best_model = model
            
    print(f"    => Selected K={best_k} for the upcoming Test Window.")
    return best_model, best_k

# ==========================================
# 5. 主程序调用示例 (接续阶段一的流水线)
# ==========================================
if __name__ == "__main__":
    db_path = 'tail_risk_data.db'
    wrds_path = '/Users/jianbinchen/NonSync/GitHub/Research/DataSet/WRDS.sqlite'
    
    # 1. 实例化阶段一的数据管道，读取真实数据并完成特征工程
    print(">>> [1/3] Loading Real Data from SQLite and Building Features...")
    # 假设 AssetPricingPipeline 已经在前文定义
    pipeline = AssetPricingPipeline(db_path=db_path)
    pipeline.load_and_build_features()
    
    # 2. 初始化 Expanding Window 生成器
    generator = pipeline.expanding_window_generator(initial_train_years=15, val_years=2, test_years=1)
    
    # 用于收集所有纯样本外预测结果的容器
    all_out_of_sample_predictions = []
    
    print("\n>>> [2/3] Starting Expanding Window MDN Training on Real Data...")
    
    # 遍历每一个时间窗口
    for window_idx, (info, train_df, val_df, test_df) in enumerate(generator):
        print(f"\n[Window {window_idx + 1}] Train: {info['train'][0]}~{info['train'][1]} | Val: {info['val'][0]}~{info['val'][1]} | Test: {info['test'][0]}~{info['test'][1]}")
        
        # 提取当前窗口的 Numpy 矩阵
        X_train = train_df[pipeline.feature_cols].values
        Y_train = train_df['target_ret_final'].values
        X_val = val_df[pipeline.feature_cols].values
        Y_val = val_df['target_ret_final'].values
        X_test = test_df[pipeline.feature_cols].values
        
        print(f"   Train samples: {len(X_train)}, Val samples: {len(X_val)}, Test samples: {len(X_test)}")
        
        # # 动态寻优寻找该窗口期下的最佳 K 值
        # best_model, best_k = find_best_mdn(X_train, Y_train, X_val, Y_val, k_candidates=[1, 3, 5])
        # print(f"   => Best K selected: {best_k}")

        # 全局固定 K=5，直接训练，极大节省算力
        best_k = 5
        print(f"   => Training MDN with fixed K={best_k}...")
        best_model, val_loss = train_mdn(X_train, Y_train, X_val, Y_val, k_components=best_k)
        
        # ----------------------------------------------------
        # 最关键的一步：对 Test 集合（样本外）进行预测并记录
        # ----------------------------------------------------
        best_model.eval()
        with torch.no_grad():
            # 移动 Test 数据到 GPU
            X_test_tensor = torch.FloatTensor(X_test).to(device)
            pi_logits, mu, sigma = best_model(X_test_tensor)
            # 将 logits 转换为概率
            pi = torch.softmax(pi_logits, dim=-1)
            
            # 回传到 CPU 并转为 Numpy 以便后续存入数据库
            pi_np = pi.cpu().numpy()
            mu_np = mu.cpu().numpy()
            sigma_np = sigma.cpu().numpy()
            
        # 构建当前窗口的预测结果表
        # 为了应对动态 K 导致的数组长度不一，我们将参数数组 JSON 序列化为字符串
        preds_df = test_df[['permno', 'date', 'target_ret_final']].copy()
        preds_df['best_k'] = best_k
        
        # json.dumps 会把 [0.2, 0.8] 变成字符串 "[0.2, 0.8]" 存入数据库
        preds_df['pi_vec'] = [json.dumps(vec.tolist()) for vec in pi_np]
        preds_df['mu_vec'] = [json.dumps(vec.tolist()) for vec in mu_np]
        preds_df['sigma_vec'] = [json.dumps(vec.tolist()) for vec in sigma_np]
        
        all_out_of_sample_predictions.append(preds_df)
        
        # (为了节约演示时间，如果你只是测试 pipeline 是否跑通，可以加上 break)
        if window_idx == 2: break 

    # 3. 将所有样本外预测结果拼接，并永久保存至 SQLite3
    print("\n>>> [3/3] Saving Out-of-Sample Predictions to SQLite3...")
    final_predictions_df = pd.concat(all_out_of_sample_predictions, ignore_index=True)
    
    conn = sqlite3.connect(db_path)
    final_predictions_df.to_sql('mdn_predictions', conn, if_exists='replace', index=False)
    
    # 建立索引，方便阶段三极速读取
    cursor = conn.cursor()
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_pred_date ON mdn_predictions (date);")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_pred_permno ON mdn_predictions (permno);")
    conn.close()
    
    print(f"=== Phase 2 Complete! Saved {len(final_predictions_df)} OOS predictions to 'mdn_predictions' table. ===")

Using device: mps
>>> [1/3] Loading Real Data from SQLite and Building Features...
Loading data from SQLite database...
Executing Feature Engineering...
Pipeline ready. Total Tensor Features: 18

>>> [2/3] Starting Expanding Window MDN Training on Real Data...

[Window 1] Train: 1990-01~2004-12 | Val: 2005-01~2006-12 | Test: 2007-01~2007-12
   Train samples: 1116347, Val samples: 114053, Test samples: 55653
   => Training MDN with fixed K=5...

[Window 2] Train: 1990-01~2005-12 | Val: 2006-01~2007-12 | Test: 2008-01~2008-12
   Train samples: 1173817, Val samples: 112236, Test samples: 53717
   => Training MDN with fixed K=5...

[Window 3] Train: 1990-01~2006-12 | Val: 2007-01~2008-12 | Test: 2009-01~2009-12
   Train samples: 1230400, Val samples: 109370, Test samples: 50068
   => Training MDN with fixed K=5...

>>> [3/3] Saving Out-of-Sample Predictions to SQLite3...
=== Phase 2 Complete! Saved 159438 OOS predictions to 'mdn_predictions' table. ===


# 结构化MDN

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader

class StructuredMDN(nn.Module):
    def __init__(self, macro_dim, micro_dim, num_components=3, hidden_dim=32):
        super(StructuredMDN, self).__init__()
        self.num_components = num_components
        
        # ---------------------------------------------------------
        # 1. Macro Network: 输入宏观特征，输出混合概率 (Regime Weights)
        # 经济学含义: 决定当前市场环境属于哪个状态的概率
        # ---------------------------------------------------------
        self.macro_net = nn.Sequential(
            nn.Linear(macro_dim, hidden_dim),
            nn.ELU(),  # ELU 比 ReLU 在金融数据中表现更平滑
            nn.Linear(hidden_dim, num_components),
            nn.Softmax(dim=-1) # 保证概率和为1
        )
        
        # ---------------------------------------------------------
        # 2. Micro Network (Mean): 输入个股特征，输出每个状态下的预期收益
        # 经济学含义: 股票的基本面在不同机制下的预期表现
        # ---------------------------------------------------------
        self.micro_net_mu = nn.Sequential(
            nn.Linear(micro_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, num_components)
        )
        
        # ---------------------------------------------------------
        # 3. Micro Network (Sigma): 输入个股特征，输出特质波动率
        # 经济学含义: 股票在不同机制下的风险敞口
        # ---------------------------------------------------------
        self.micro_net_sigma = nn.Sequential(
            nn.Linear(micro_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, num_components)
        )

    def forward(self, x_macro, x_micro):
        # 1. 计算混合权重 \pi (形状: [batch_size, num_components])
        pi = self.macro_net(x_macro)
        
        # 2. 计算均值 \mu (形状: [batch_size, num_components])
        mu = self.micro_net_mu(x_micro)
        
        # 3. 计算标准差 \sigma (形状: [batch_size, num_components])
        # 注意: 必须保证 sigma > 0。Softplus 比 exp 更稳定，防止梯度爆炸
        sigma = F.softplus(self.micro_net_sigma(x_micro)) + 1e-6 
        
        return pi, mu, sigma

In [ ]:
def mdn_loss(pi, mu, sigma, target):
    """
    计算 MDN 的负对数似然 (NLL) 损失。
    使用 LogSumExp 技巧保证数值绝对稳定。
    """
    # 扩展 target 的维度以匹配 mu 和 sigma [batch_size, num_components]
    target = target.view(-1, 1).expand_as(mu)
    
    # 构造高斯分布对象
    m = torch.distributions.Normal(loc=mu, scale=sigma)
    
    # 计算 log(\mathcal{N}(y | \mu, \sigma))
    log_prob = m.log_prob(target)
    
    # 计算 log(\pi)。加上 1e-8 防止 log(0)
    log_pi = torch.log(pi + 1e-8)
    
    # 核心步骤: log(\sum \pi_k * \mathcal{N}) = logsumexp(log(\pi_k) + log(\mathcal{N}))
    # 这样可以避免直接计算指数导致的溢出
    loss = -torch.logsumexp(log_pi + log_prob, dim=1)
    
    return loss.mean()

In [ ]:
class AssetPricingDataset(Dataset):
    def __init__(self, crsp_df, ratios_df, macro_df, micro_cols, macro_cols):
        # 1. 假设你已经通过 merge_asof 对齐了时间，并做好了 mthret_lead1 (下月收益率)
        # final_df = pd.merge_asof(...) 
        
        # 2. 宏观特征标准化 (Z-score)
        self.X_macro = torch.tensor(
            final_df[macro_cols].apply(lambda x: (x - x.mean()) / x.std()).values, 
            dtype=torch.float32
        )
        
        # 3. 微观特征【截面秩转换】(Rank Normalization) -> 压入 [-1, 1]
        # 这一步极其重要，防止 WRDS Ratios 的极值摧毁网络
        def rank_norm(group):
            return (group.rank() - 1) / (len(group) - 1) * 2 - 1
            
        ranked_micro = final_df.groupby('mthcaldt')[micro_cols].apply(rank_norm)
        self.X_micro = torch.tensor(ranked_micro.values, dtype=torch.float32)
        
        # 4. 目标变量 Y (下月收益率，保持真实数值，千万别缩尾！)
        self.Y = torch.tensor(final_df['mthret_lead1'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        return self.X_macro[idx], self.X_micro[idx], self.Y[idx]

# 设定你挑选的特征列表
macro_features = ['d/p', 'e/p', 'svar', 'tail', 'vixcls'] # Goyal宏观
micro_features = ['bm', 'roe', 'ocf_lct', 'short_debt', 'accrual'] # WRDS微观 + CRSP动量

In [ ]:
# 1. 初始化模型
num_macro = len(macro_features)
num_micro = len(micro_features)
num_components = 3  # 假设3个状态: 崩盘(左尾), 震荡(均值), 狂热(右尾)

model = StructuredMDN(macro_dim=num_macro, micro_dim=num_micro, num_components=num_components)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4) # L2 正则化防过拟合

# 2. 假设 dataset 和 dataloader 已经准备好
# dataloader = DataLoader(dataset, batch_size=1024, shuffle=True)

epochs = 50
model.train()

for epoch in range(epochs):
    total_loss = 0
    for batch_macro, batch_micro, batch_y in dataloader:
        optimizer.zero_grad()
        
        # 前向传播
        pi, mu, sigma = model(batch_macro, batch_micro)
        
        # 计算 Loss
        loss = mdn_loss(pi, mu, sigma, batch_y)
        
        # 反向传播
        loss.backward()
        
        # 梯度裁剪 (Gradient Clipping) -> 金融深度学习的保命符！
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(dataloader):.4f}")

In [3]:
import pandas as pd
import numpy as np
import sqlite3
import os
pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行
# list the tables in the database to verify
wrds_path = '/Users/jianbinchen/NonSync/GitHub/Research/DataSet/WRDS.sqlite'
# set current working directory to the location of the database file
os.chdir('/Users/jianbinchen/NonSync/GitHub/Research/DCDE')

file_path='/Users/jianbinchen/NonSync/GitHub/Research/DCDE/data/gwz_Data2024.xlsx'
df = pd.read_excel(file_path, sheet_name='Monthly')
# read the micro and macro data
micro_df=pd.read_parquet('./data/micro_data.parquet')
macro_df=pd.read_parquet('./data/macro_data.parquet')

In [4]:
micro_df = pd.read_parquet('./data/micro_data.parquet')
micro_df.columns

Index(['permno', 'mthcaldt', 'mthret', 'mthprc', 'mthcap', 'mthvol', 'shrout',
       'short_debt', 'intcov_ratio', 'debt_at', 'debt_ebitda', 'ocf_lct',
       'cash_ratio', 'accrual', 'bm', 'pe_exi', 'evm', 'divyield', 'roe',
       'gpm', 'at_turn', 'rd_sale', 'mthret_lead1', 'turnover_1m',
       'mthcap_log', 'MOM12m'],
      dtype='object')

In [5]:
macro_df = pd.read_parquet('./data/macro_data.parquet')
macro_df.columns

Index(['date', 'AAA', 'BAA', 'lty', 'ltr', 'tbl', 'd/p', 'd/y', 'e/p', 'd/e',
       'b/m', 'ntis', 'svar', 'disag', 'skvw', 'tail', 'rdsp', 'avgcor', 'tms',
       'dfy', 'dfr', 'infl', 'vrp'],
      dtype='object')

In [8]:
micro_cols=micro_df.columns.drop(['mthcaldt', 'permno'])
macro_cols=macro_df.columns.drop('date')

In [9]:
micro_cols=micro_df.columns.drop(['mthcaldt', 'permno'])
micro_cols

Index(['mthret', 'mthprc', 'mthcap', 'mthvol', 'shrout', 'short_debt',
       'intcov_ratio', 'debt_at', 'debt_ebitda', 'ocf_lct', 'cash_ratio',
       'accrual', 'bm', 'pe_exi', 'evm', 'divyield', 'roe', 'gpm', 'at_turn',
       'rd_sale', 'mthret_lead1', 'turnover_1m', 'mthcap_log', 'MOM12m'],
      dtype='object')

In [13]:
type(macro_cols)

pandas.core.indexes.base.Index

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import json
import copy
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行

# ==========================================
# 0. 自动设备检测
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# ==========================================
# 1. 外置数据预处理模块 (ETL & Feature Engineering)
# ==========================================
print(">>> [1/3] Loading and Preprocessing Data (Externalized)...")

micro_df = pd.read_parquet('./data/micro_data.parquet')
macro_df = pd.read_parquet('./data/macro_data.parquet')
micro_cols=micro_df.columns.drop(['mthcaldt', 'permno'])
macro_cols=macro_df.columns.drop('date')

# 1.1 日期对齐与合并
micro_df['mthcaldt'] = pd.to_datetime(micro_df['mthcaldt']) + pd.offsets.MonthEnd(0)
macro_df['date'] = pd.to_datetime(macro_df['date']) + pd.offsets.MonthEnd(0)

df = pd.merge(micro_df, macro_df, left_on='mthcaldt', right_on='date', how='inner')
df = df.sort_values(['permno', 'mthcaldt']).reset_index(drop=True)

# 1.2 目标收益率设定 (绝对禁止缩尾！)
df['target_ret_final'] = df['mthret_lead1']

# 1.3 缺失值预填充 (可在此处放置你的业务填充逻辑)
# 例如：使用截面均值填充微观特征缺失值
for feature in micro_cols:
    if feature in df.columns:
        df[feature] = df[feature].fillna(df.groupby('mthcaldt')[feature].transform('mean'))

# 1.4 微观特征：截面秩标准化 (Cross-Sectional Rank Normalization) [-1, 1]
print(" -> Applying Rank Normalization to Micro Features...")
micro_tensor_cols = []
for col in micro_cols:
    norm_col = f'{col}_norm'
    # 使用 pct=True 转化为 0~1，再映射到 -1~1
    df[norm_col] = df.groupby('mthcaldt')[col].transform(
        lambda x: (x.rank(pct=True) * 2) - 1
    ).fillna(0)  # 如果全截面缺失，填0(中性)
    micro_tensor_cols.append(norm_col)

# 1.5 宏观特征：时间序列 Z-score 标准化
print(" -> Applying Z-score Normalization to Macro Features...")
macro_tensor_cols = []
for col in macro_cols:
    norm_col = f'{col}_norm'
    df[norm_col] = (df[col] - df[col].mean()) / (df[col].std() + 1e-8)
    df[norm_col] = df[norm_col].fillna(0)
    macro_tensor_cols.append(norm_col)

# 1.6 清理最终的空值行并固化数据集
processed_df = df.dropna(subset=['target_ret_final'] + micro_tensor_cols + macro_tensor_cols).copy()
print(f"Data preprocessing complete! Final shape: {processed_df.shape}")
print(f"Micro Features: {len(micro_tensor_cols)}, Macro Features: {len(macro_tensor_cols)}")

# (可选) 你可以在这里将 processed_df 存为 Parquet:
# processed_df.to_parquet('./data/fully_processed_model_input.parquet')


# ==========================================
# 2. 轻量级时间窗口生成器
# ==========================================
class DataWindowGenerator:
    """
    仅负责按时间切分数据，向模型输送 Train/Val/Test 窗口
    """
    def __init__(self, df, date_col='mthcaldt'):
        self.df = df
        self.date_col = date_col
        
    def expanding_window_generator(self, initial_train_years=20, val_years=2, test_years=1):
        dates = np.sort(self.df[self.date_col].unique())
        initial_train_months = initial_train_years * 12
        val_months = val_years * 12
        test_months = test_years * 12
        
        start_idx = 0
        current_split_idx = initial_train_months
        
        while current_split_idx + val_months + test_months <= len(dates):
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data = self.df[self.df[self.date_col].isin(val_dates)]
            test_data = self.df[self.df[self.date_col].isin(test_dates)]
            
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            current_split_idx += test_months

# ==========================================
# 3. 结构化 MDN (Structured MoE-MDN)
# ==========================================
class StructuredMDN(nn.Module):
    def __init__(self, macro_dim, micro_dim, hidden_dim=64, num_components=3):
        super(StructuredMDN, self).__init__()
        self.num_components = num_components
        
        # 通道 A: Macro Network (门控网络) -> 预测机制权重 pi
        self.macro_net = nn.Sequential(
            nn.Linear(macro_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ELU(),
            nn.Linear(hidden_dim // 2, num_components)
        )
        
        # 通道 B: Micro Network (专家网络) -> 预测特质 mu 和 sigma
        self.micro_extractor = nn.Sequential(
            nn.Linear(micro_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ELU()
        )
        
        self.mu_head = nn.Linear(hidden_dim // 2, num_components)
        self.sigma_head = nn.Linear(hidden_dim // 2, num_components)

    def forward(self, x_macro, x_micro):
        pi_logits = self.macro_net(x_macro)
        h_micro = self.micro_extractor(x_micro)
        mu = self.mu_head(h_micro)
        sigma = nn.functional.softplus(self.sigma_head(h_micro)) + 1e-6 
        return pi_logits, mu, sigma
    
# ==========================================
# 4. 稳健的 NLL 损失函数 (支持时间衰减权重)
# ==========================================
def mdn_nll_loss_weighted(pi_logits, mu, sigma, target, weights):
    """
    weights: shape (batch_size, 1)，距离现在越近权重大
    """
    log_pi = torch.log_softmax(pi_logits, dim=-1)
    normal_dist = torch.distributions.Normal(mu, sigma)
    log_normal = normal_dist.log_prob(target.expand_as(mu))
    log_mix = log_pi + log_normal
    
    # 每个样本基础的 NLL loss
    loss = -torch.logsumexp(log_mix, dim=-1) 
    
    # 乘以时间衰减权重后求平均
    # 确保 weights 的 shape 与 loss 对齐
    weighted_loss = loss * weights.squeeze()
    
    return weighted_loss.mean()

# ==========================================
# 5. 双通道训练循环 (支持 Warm Start 与 Weights)
# ==========================================
def train_structured_mdn(model,  # 【修改点1】模型从外部传入，实现热启动
                         X_train_macro, X_train_micro, y_train, w_train, # 【修改点2】加入训练权重
                         X_val_macro, X_val_micro, y_val, w_val,         # 【修改点2】加入验证权重
                         epochs=500, batch_size=512, lr=1e-3, patience=20):
    
    # 每次进入新窗口，重新初始化 Optimizer，清除历史动量缓存，防止陷入局部最优
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # 将权重 w_train 和 w_val 打包进 TensorDataset
    tr_dataset = TensorDataset(torch.FloatTensor(X_train_macro), 
                               torch.FloatTensor(X_train_micro), 
                               torch.FloatTensor(y_train).unsqueeze(1),
                               torch.FloatTensor(w_train).unsqueeze(1))
    
    va_dataset = TensorDataset(torch.FloatTensor(X_val_macro), 
                               torch.FloatTensor(X_val_micro), 
                               torch.FloatTensor(y_val).unsqueeze(1),
                               torch.FloatTensor(w_val).unsqueeze(1))
    
    train_loader = DataLoader(tr_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(va_dataset, batch_size=batch_size, shuffle=False)
    
    best_val_loss = float('inf')
    best_model_state = None
    epochs_no_improve = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for b_macro, b_micro, b_y, b_w in train_loader:
            b_macro, b_micro, b_y, b_w = b_macro.to(device), b_micro.to(device), b_y.to(device), b_w.to(device)
            optimizer.zero_grad()
            
            pi_logits, mu, sigma = model(b_macro, b_micro)
            # 【修改点3】调用带权重的 Loss
            loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, b_y, b_w)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * b_macro.size(0)
            
        train_loss /= len(train_loader.dataset)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for b_macro, b_micro, b_y, b_w in val_loader:
                b_macro, b_micro, b_y, b_w = b_macro.to(device), b_micro.to(device), b_y.to(device), b_w.to(device)
                pi_logits, mu, sigma = model(b_macro, b_micro)
                loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, b_y, b_w)
                val_loss += loss.item() * b_macro.size(0)
        val_loss /= len(val_loader.dataset)
            
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break
                
    model.load_state_dict(best_model_state)
    return model, best_val_loss


# ==========================================
# 6. 启动模型训练流水线
# ==========================================
def calculate_time_decay_weights(df_dates, half_life_months=36):
    """
    计算基于半衰期的指数衰减权重。
    参数：
    df_dates: pd.Series, 包含 mthcaldt 日期
    half_life_months: 半衰期(月)。默认 36 个月(3年)权重减半。
    """
    max_date = df_dates.max()
    # 计算每个样本距离该窗口最新日期的月数差
    months_diff = (max_date.year - df_dates.dt.year) * 12 + (max_date.month - df_dates.dt.month)
    
    # 衰减因子 lambda: (0.5)^(1/half_life)
    decay_factor = np.power(0.5, 1.0 / half_life_months)
    
    # 计算基础权重
    weights = np.power(decay_factor, months_diff).values
    
    # 【极度关键】归一化权重，使得权重的均值为 1。
    # 如果不归一化，整个 Loss 的绝对值会变小，导致梯度的步长变相缩小！
    weights = weights / weights.mean() 
    return weights

if __name__ == "__main__":
    print("\n>>> [2/3] Initializing Data Window Generator...")
    
    generator = DataWindowGenerator(df=processed_df, date_col='mthcaldt')
    window_gen = generator.expanding_window_generator(initial_train_years=15, val_years=2, test_years=1)
    
    all_oos_predictions = []
    
    # 【修改点4：热启动核心】在循环外全局初始化模型，让它继承历史记忆！
    best_k = 3
    macro_dim = len(macro_tensor_cols)
    micro_dim = len(micro_tensor_cols)
    global_model = StructuredMDN(macro_dim, micro_dim, num_components=best_k).to(device)
    
    print("\n>>> [3/3] Starting Expanding Window Structured MDN Training (Warm Start & Time Decay)...")
    for window_idx, (info, train_df, val_df, test_df) in enumerate(window_gen):
        print(f"\n[Window {window_idx + 1}] Train: {info['train'][0]}~{info['train'][1]} | Test: {info['test'][0]}~{info['test'][1]}")
        
        # --- 计算时间衰减权重 ---
        w_train = calculate_time_decay_weights(train_df['mthcaldt'], half_life_months=36)
        w_val = calculate_time_decay_weights(val_df['mthcaldt'], half_life_months=12) # 验证集更看重近期，半衰期设短一点
        
        X_tr_mac, X_tr_mic = train_df[macro_tensor_cols].values, train_df[micro_tensor_cols].values
        Y_tr = train_df['target_ret_final'].values
        
        X_va_mac, X_va_mic = val_df[macro_tensor_cols].values, val_df[micro_tensor_cols].values
        Y_va = val_df['target_ret_final'].values
        
        X_te_mac, X_te_mic = test_df[macro_tensor_cols].values, test_df[micro_tensor_cols].values
        
        # --- 动态学习率 ---
        # 第一轮窗口是从零开始(Cold Start)，学习率正常；
        # 后续窗口由于有 Warm Start，相当于微调，学习率降低防止破坏已有知识
        current_lr = 1e-3 if window_idx == 0 else 1e-4
        
        global_model, val_loss = train_structured_mdn(
            model=global_model,     # 传入全局模型
            X_train_macro=X_tr_mac, X_train_micro=X_tr_mic, y_train=Y_tr, w_train=w_train,
            X_val_macro=X_va_mac, X_val_micro=X_va_mic, y_val=Y_va, w_val=w_val,
            k_components=best_k,
            lr=current_lr           # 使用动态学习率
        )
        
        # --- 样本外预测 ---
        global_model.eval()
        with torch.no_grad():
            pi_logits, mu, sigma = global_model(
                torch.FloatTensor(X_te_mac).to(device), 
                torch.FloatTensor(X_te_mic).to(device)
            )
            pi = torch.softmax(pi_logits, dim=-1)
            
        preds_df = test_df[['permno', 'mthcaldt', 'target_ret_final']].copy()
        preds_df['pi_vec'] = [json.dumps(vec.tolist()) for vec in pi.cpu().numpy()]
        preds_df['mu_vec'] = [json.dumps(vec.tolist()) for vec in mu.cpu().numpy()]
        preds_df['sigma_vec'] = [json.dumps(vec.tolist()) for vec in sigma.cpu().numpy()]
        all_oos_predictions.append(preds_df)
        
        # 测试用，仅跑 1 个窗口
        # if window_idx == 0: break

    print("\n>>> Pipeline complete. You can now use Monte Carlo on (pi, mu, sigma) to compute ES_5%!")

In [ ]:

# ==========================================
# 4. 稳健的 NLL 损失函数
# ==========================================
def mdn_nll_loss(pi_logits, mu, sigma, target):
    log_pi = torch.log_softmax(pi_logits, dim=-1)
    normal_dist = torch.distributions.Normal(mu, sigma)
    log_normal = normal_dist.log_prob(target.expand_as(mu))
    log_mix = log_pi + log_normal
    loss = -torch.logsumexp(log_mix, dim=-1) 
    return loss.mean()



# ==========================================
# 5. 双通道训练循环
# ==========================================
def train_structured_mdn(X_train_macro, X_train_micro, y_train, 
                         X_val_macro, X_val_micro, y_val, 
                         k_components, epochs=500, batch_size=512, lr=1e-3, patience=20):
    
    macro_dim = X_train_macro.shape[1]
    micro_dim = X_train_micro.shape[1]
    
    model = StructuredMDN(macro_dim, micro_dim, num_components=k_components).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    tr_dataset = TensorDataset(torch.FloatTensor(X_train_macro), torch.FloatTensor(X_train_micro), torch.FloatTensor(y_train).unsqueeze(1))
    va_dataset = TensorDataset(torch.FloatTensor(X_val_macro), torch.FloatTensor(X_val_micro), torch.FloatTensor(y_val).unsqueeze(1))
    
    train_loader = DataLoader(tr_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(va_dataset, batch_size=batch_size, shuffle=False)
    
    best_val_loss = float('inf')
    best_model_state = None
    epochs_no_improve = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for b_macro, b_micro, b_y in train_loader:
            b_macro, b_micro, b_y = b_macro.to(device), b_micro.to(device), b_y.to(device)
            optimizer.zero_grad()
            
            pi_logits, mu, sigma = model(b_macro, b_micro)
            loss = mdn_nll_loss(pi_logits, mu, sigma, b_y)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * b_macro.size(0)
            
        train_loss /= len(train_loader.dataset)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for b_macro, b_micro, b_y in val_loader:
                b_macro, b_micro, b_y = b_macro.to(device), b_micro.to(device), b_y.to(device)
                pi_logits, mu, sigma = model(b_macro, b_micro)
                loss = mdn_nll_loss(pi_logits, mu, sigma, b_y)
                val_loss += loss.item() * b_macro.size(0)
        val_loss /= len(val_loader.dataset)
            
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break
                
    model.load_state_dict(best_model_state)
    return model, best_val_loss

# ==========================================
# 6. 启动模型训练流水线
# ==========================================
if __name__ == "__main__":
    print("\n>>> [2/3] Initializing Data Window Generator...")
    
    # 直接将处理好的 processed_df 喂给生成器
    generator = DataWindowGenerator(df=processed_df, date_col='mthcaldt')
    window_gen = generator.expanding_window_generator(initial_train_years=15, val_years=2, test_years=1)
    
    all_oos_predictions = []
    
    print("\n>>> [3/3] Starting Expanding Window Structured MDN Training...")
    for window_idx, (info, train_df, val_df, test_df) in enumerate(window_gen):
        print(f"\n[Window {window_idx + 1}] Train: {info['train'][0]}~{info['train'][1]} | Test: {info['test'][0]}~{info['test'][1]}")
        
        # 从处理好的 df 中分离出所需的输入和输出
        X_tr_mac, X_tr_mic = train_df[macro_tensor_cols].values, train_df[micro_tensor_cols].values
        Y_tr = train_df['target_ret_final'].values
        
        X_va_mac, X_va_mic = val_df[macro_tensor_cols].values, val_df[micro_tensor_cols].values
        Y_va = val_df['target_ret_final'].values
        
        X_te_mac, X_te_mic = test_df[macro_tensor_cols].values, test_df[micro_tensor_cols].values
        
        best_k = 3 # 3个状态(崩盘、震荡、繁荣)
        best_model, val_loss = train_structured_mdn(
            X_tr_mac, X_tr_mic, Y_tr, 
            X_va_mac, X_va_mic, Y_va, 
            k_components=best_k
        )
        
        # 样本外预测
        best_model.eval()
        with torch.no_grad():
            pi_logits, mu, sigma = best_model(
                torch.FloatTensor(X_te_mac).to(device), 
                torch.FloatTensor(X_te_mic).to(device)
            )
            pi = torch.softmax(pi_logits, dim=-1)
            
        preds_df = test_df[['permno', 'mthcaldt', 'target_ret_final']].copy()
        preds_df['pi_vec'] = [json.dumps(vec.tolist()) for vec in pi.cpu().numpy()]
        preds_df['mu_vec'] = [json.dumps(vec.tolist()) for vec in mu.cpu().numpy()]
        preds_df['sigma_vec'] = [json.dumps(vec.tolist()) for vec in sigma.cpu().numpy()]
        all_oos_predictions.append(preds_df)
        
        # 测试用，仅跑 1 个窗口
        if window_idx == 0: break

    print("\n>>> Pipeline complete. You can now use Monte Carlo on (pi, mu, sigma) to compute ES_5%!")

In [ ]:
# Normalize macro features (Z-score)
# 3. 微观特征：截面秩标准化 (Cross-Sectional Rank Normalization) [-1, 1]
print(" -> Applying Rank Normalization to Micro Features...")
for col in self.raw_micro_cols:
    norm_col = f'{col}_norm'
    df[norm_col] = df.groupby('mthcaldt')[col].transform(
        lambda x: (x.rank(pct=True) * 2) - 1
    ).fillna(0)
    self.micro_tensor_cols.append(norm_col)
# 4. 宏观特征：时间序列 Z-score 标准化
print(" -> Applying Z-score Normalization to Macro Features...")
for col in self.raw_macro_cols:
    norm_col = f'{col}_norm'
    # 简单全局标准化 (由于宏观指标较为平稳，且只作为门控信号)
    df[norm_col] = (df[col] - df[col].mean()) / (df[col].std() + 1e-8)
    df[norm_col] = df[norm_col].fillna(0)
    self.macro_tensor_cols.append(norm_col)



In [ ]:
# 1. 对齐宏观与微观 (通过 date 和 mthcaldt 对齐)
self.df_micro['mthcaldt'] = pd.to_datetime(self.df_micro['mthcaldt'])+ pd.offsets.MonthEnd(0)
self.df_macro['date'] = pd.to_datetime(self.df_macro['date'])+ pd.offsets.MonthEnd(0)

df = pd.merge(self.df_micro, self.df_macro, left_on='mthcaldt', right_on='date', how='inner')
df = df.sort_values(['permno', 'mthcaldt']).reset_index(drop=True)

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import json
import copy
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 0. 自动设备检测
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# ==========================================
# 1. 结构化数据管道 (AssetPricingPipeline)
# ==========================================
class AssetPricingPipeline:
    def __init__(self, merged_micro_df, macro_df, micro_features_list, macro_features_list):
        self.df_micro = merged_micro_df.copy()
        self.df_macro = macro_df.copy()
        self.raw_micro_cols = micro_features_list
        self.raw_macro_cols = macro_features_list
        
        self.micro_tensor_cols = []
        self.macro_tensor_cols = []
        self.df = None
        
    def build_structured_features(self):
        print("Executing Structured Feature Engineering...")
        
        # 1. 对齐宏观与微观 (通过 date 和 mthcaldt 对齐)
        self.df_micro['mthcaldt'] = pd.to_datetime(self.df_micro['mthcaldt'])+ pd.offsets.MonthEnd(0)
        self.df_macro['date'] = pd.to_datetime(self.df_macro['date'])+ pd.offsets.MonthEnd(0)
        
        df = pd.merge(self.df_micro, self.df_macro, left_on='mthcaldt', right_on='date', how='inner')
        df = df.sort_values(['permno', 'mthcaldt']).reset_index(drop=True)
        
        # 2. 目标收益率 (TARGET) —— 【绝对禁止缩尾 WINSORIZE! 保留原汁原味的尾部极值】
        df['target_ret_final'] = df['mthret_lead1'] 
        
        # 3. 微观特征：截面秩标准化 (Cross-Sectional Rank Normalization) [-1, 1]
        print(" -> Applying Rank Normalization to Micro Features...")
        for col in self.raw_micro_cols:
            norm_col = f'{col}_norm'
            df[norm_col] = df.groupby('mthcaldt')[col].transform(
                lambda x: (x.rank(pct=True) * 2) - 1
            ).fillna(0)
            self.micro_tensor_cols.append(norm_col)
            
        # 4. 宏观特征：时间序列 Z-score 标准化
        print(" -> Applying Z-score Normalization to Macro Features...")
        for col in self.raw_macro_cols:
            norm_col = f'{col}_norm'
            # 简单全局标准化 (由于宏观指标较为平稳，且只作为门控信号)
            df[norm_col] = (df[col] - df[col].mean()) / (df[col].std() + 1e-8)
            df[norm_col] = df[norm_col].fillna(0)
            self.macro_tensor_cols.append(norm_col)
            
        # 5. 清理缺失值
        self.df = df.dropna(subset=['target_ret_final'] + self.micro_tensor_cols + self.macro_tensor_cols)
        print(f"Pipeline ready. Micro Features: {len(self.micro_tensor_cols)}, Macro Features: {len(self.macro_tensor_cols)}")

    def expanding_window_generator(self, initial_train_years=20, val_years=2, test_years=1):
        dates = np.sort(self.df['mthcaldt'].unique())
        initial_train_months = initial_train_years * 12
        val_months = val_years * 12
        test_months = test_years * 12
        
        start_idx = 0
        current_split_idx = initial_train_months
        
        while current_split_idx + val_months + test_months <= len(dates):
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            train_data = self.df[self.df['mthcaldt'].isin(train_dates)]
            val_data = self.df[self.df['mthcaldt'].isin(val_dates)]
            test_data = self.df[self.df['mthcaldt'].isin(test_dates)]
            
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            current_split_idx += test_months

# ==========================================
# 2. 结构化 MDN (Structured MoE-MDN)
# ==========================================
class StructuredMDN(nn.Module):
    def __init__(self, macro_dim, micro_dim, hidden_dim=64, num_components=3):
        super(StructuredMDN, self).__init__()
        self.num_components = num_components
        
        # ---------------------------------------------------------
        # 通道 A: Macro Network (门控网络) -> 预测机制权重 pi
        # ---------------------------------------------------------
        self.macro_net = nn.Sequential(
            nn.Linear(macro_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2), # 宏观截面相同，必须用 LayerNorm 防止崩溃
            nn.ELU(),
            # nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, num_components) # 输出 pi_logits
        )
        
        # ---------------------------------------------------------
        # 通道 B: Micro Network (专家网络) -> 预测特质 mu 和 sigma
        # ---------------------------------------------------------
        self.micro_extractor = nn.Sequential(
            nn.Linear(micro_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ELU(),
            # nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ELU()
        )
        
        # 专家输出头
        self.mu_head = nn.Linear(hidden_dim // 2, num_components)
        self.sigma_head = nn.Linear(hidden_dim // 2, num_components)

    def forward(self, x_macro, x_micro):
        # 宏观决定混合权重 (Regime Probabilities)
        pi_logits = self.macro_net(x_macro)
        
        # 微观决定分布参数 (Component Moments)
        h_micro = self.micro_extractor(x_micro)
        mu = self.mu_head(h_micro)
        
        # 使用 Softplus 保证方差严谨为正，且比 ELU+1 更平滑防爆破
        sigma = nn.functional.softplus(self.sigma_head(h_micro)) + 1e-6 
        
        return pi_logits, mu, sigma

# ==========================================
# 3. 稳健的 NLL 损失函数
# ==========================================
def mdn_nll_loss(pi_logits, mu, sigma, target):
    log_pi = torch.log_softmax(pi_logits, dim=-1) # (batch, K)
    normal_dist = torch.distributions.Normal(mu, sigma)
    log_normal = normal_dist.log_prob(target.expand_as(mu))
    log_mix = log_pi + log_normal
    loss = -torch.logsumexp(log_mix, dim=-1) 
    return loss.mean()

# ==========================================
# 4. 双通道训练循环
# ==========================================
def train_structured_mdn(X_train_macro, X_train_micro, y_train, 
                         X_val_macro, X_val_micro, y_val, 
                         k_components, epochs=500, batch_size=512, lr=1e-3, patience=20):
    
    macro_dim = X_train_macro.shape[1]
    micro_dim = X_train_micro.shape[1]
    
    model = StructuredMDN(macro_dim, micro_dim, num_components=k_components).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # 包装为 TensorDataset (注意现在有 3 个输入)
    tr_dataset = TensorDataset(torch.FloatTensor(X_train_macro), torch.FloatTensor(X_train_micro), torch.FloatTensor(y_train).unsqueeze(1))
    va_dataset = TensorDataset(torch.FloatTensor(X_val_macro), torch.FloatTensor(X_val_micro), torch.FloatTensor(y_val).unsqueeze(1))
    
    train_loader = DataLoader(tr_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(va_dataset, batch_size=batch_size, shuffle=False)
    
    best_val_loss = float('inf')
    best_model_state = None
    epochs_no_improve = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for b_macro, b_micro, b_y in train_loader:
            b_macro, b_micro, b_y = b_macro.to(device), b_micro.to(device), b_y.to(device)
            optimizer.zero_grad()
            
            pi_logits, mu, sigma = model(b_macro, b_micro)
            loss = mdn_nll_loss(pi_logits, mu, sigma, b_y)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * b_macro.size(0)
            
        train_loss /= len(train_loader.dataset)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for b_macro, b_micro, b_y in val_loader:
                b_macro, b_micro, b_y = b_macro.to(device), b_micro.to(device), b_y.to(device)
                pi_logits, mu, sigma = model(b_macro, b_micro)
                loss = mdn_nll_loss(pi_logits, mu, sigma, b_y)
                val_loss += loss.item() * b_macro.size(0)
        val_loss /= len(val_loader.dataset)
            
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break
                
    model.load_state_dict(best_model_state)
    return model, best_val_loss

# ==========================================
# 5. 启动流水线
# ==========================================
if __name__ == "__main__":
    # 定义输入列表 (确保前面 df 里有这些列名)
    micro_cols = features_to_fill + ['mthret', 'turnover_1m', 'mthcap_log', 'MOM12m'] 
    macro_cols = required_cols 

    print(">>> [1/3] Initializing Structured Pipeline...")
    # 这里直接传入你上半部分处理好的 merged_df 和 macro_df
    pipeline = AssetPricingPipeline(merged_micro_df=merged_df, 
                                    macro_df=macro_df, 
                                    micro_features_list=micro_cols, 
                                    macro_features_list=macro_cols)
    pipeline.build_structured_features()
    
    generator = pipeline.expanding_window_generator(initial_train_years=15, val_years=2, test_years=1)
    all_oos_predictions = []
    
    print("\n>>> [2/3] Starting Expanding Window Structured MDN Training...")
    for window_idx, (info, train_df, val_df, test_df) in enumerate(generator):
        print(f"\n[Window {window_idx + 1}] Train: {info['train'][0]}~{info['train'][1]} | Test: {info['test'][0]}~{info['test'][1]}")
        
        # 分离宏观与微观特征
        X_tr_mac, X_tr_mic = train_df[pipeline.macro_tensor_cols].values, train_df[pipeline.micro_tensor_cols].values
        Y_tr = train_df['target_ret_final'].values
        
        X_va_mac, X_va_mic = val_df[pipeline.macro_tensor_cols].values, val_df[pipeline.micro_tensor_cols].values
        Y_va = val_df['target_ret_final'].values
        
        X_te_mac, X_te_mic = test_df[pipeline.macro_tensor_cols].values, test_df[pipeline.micro_tensor_cols].values
        
        best_k = 3 # 结构化后 K 不用太大，3 个状态(崩盘、震荡、繁荣)通常足矣
        best_model, val_loss = train_structured_mdn(
            X_tr_mac, X_tr_mic, Y_tr, 
            X_va_mac, X_va_mic, Y_va, 
            k_components=best_k
        )
        
        # 样本外预测
        best_model.eval()
        with torch.no_grad():
            pi_logits, mu, sigma = best_model(
                torch.FloatTensor(X_te_mac).to(device), 
                torch.FloatTensor(X_te_mic).to(device)
            )
            pi = torch.softmax(pi_logits, dim=-1)
            
        preds_df = test_df[['permno', 'mthcaldt', 'target_ret_final']].copy()
        preds_df['pi_vec'] = [json.dumps(vec.tolist()) for vec in pi.cpu().numpy()]
        preds_df['mu_vec'] = [json.dumps(vec.tolist()) for vec in mu.cpu().numpy()]
        preds_df['sigma_vec'] = [json.dumps(vec.tolist()) for vec in sigma.cpu().numpy()]
        all_oos_predictions.append(preds_df)
        
        # 测试用，仅跑 1 个窗口
        if window_idx == 0: break

    print("\n>>> [3/3] Prediction complete. You can now use Monte Carlo on (pi, mu, sigma) to compute ES_5%!")

Using device: mps
>>> [1/3] Initializing Structured Pipeline...
Executing Structured Feature Engineering...
 -> Applying Rank Normalization to Micro Features...
 -> Applying Z-score Normalization to Macro Features...
Pipeline ready. Micro Features: 19, Macro Features: 21

>>> [2/3] Starting Expanding Window Structured MDN Training...

[Window 1] Train: 1990-01~2004-12 | Test: 2007-01~2007-12

>>> [3/3] Prediction complete. You can now use Monte Carlo on (pi, mu, sigma) to compute ES_5%!
